<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/Recurrent_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Vanille RNN

In [ ]:
from datasets import load_dataset
from collections import Counter
import re


In [36]:
dataset = load_dataset("glue", "sst2")

print(dataset)
print(dataset["train"][0])
print(dataset["train"].features)

train_data = dataset["train"].shuffle(seed=42).select(range(5000))
valid_data = dataset["validation"].shuffle(seed=42)

print("Train labels:", Counter(train_data["label"]))
print("Valid labels:", Counter(valid_data["label"]))

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
{'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'idx': Value('int32')}
Train labels: Counter({1: 2734, 0: 2266})
Valid labels: Counter({1: 444, 0: 428})


In [37]:
def tokenize(text):
    return re.findall(r"\b\w+\b|[!?.,]", text.lower())


counter = Counter()

for item in train_data:
    tokens = tokenize(item["sentence"])
    counter.update(tokens)


vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

min_freq = 2

for word, count in counter.items():
    if count >= min_freq:
        vocab[word] = len(vocab)

print("Vocab size:", len(vocab))
print(list(vocab.items())[:20])

Vocab size: 3949
[('<PAD>', 0), ('<UNK>', 1), (',', 2), ('charming', 3), ('in', 4), ('comedies', 5), ('like', 6), ('american', 7), ('and', 8), ('dead', 9), ('on', 10), ('be', 11), ('soulful', 12), ('the', 13), ('proud', 14), ('warrior', 15), ('that', 16), ('still', 17), ('souls', 18), ('of', 19)]


In [38]:
max_length = 30

def encode_text(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return ids


def pad_or_truncate(ids, max_length):
    if len(ids) > max_length:
        ids = ids[:max_length]

    return ids + [vocab["<PAD>"]] * (max_length - len(ids))

In [39]:
class SentimentDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_length):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["sentence"]
        label = self.dataset[idx]["label"]

        ids = encode_text(text, self.vocab)

        length = min(len(ids), self.max_length)

        ids = pad_or_truncate(ids, self.max_length)

        X = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        length = torch.tensor(length, dtype=torch.long)

        return X, y, length

In [41]:
train_set = SentimentDataset(train_data, vocab, max_length)
valid_set = SentimentDataset(valid_data, vocab, max_length)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64)

X_batch, y_batch, lengths_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("lengths_batch shape:", lengths_batch.shape)
print("Example X:", X_batch[0])
print("Example y:", y_batch[0])
print("Example lengths:", lengths_batch[0])

X_batch shape: torch.Size([64, 30])
y_batch shape: torch.Size([64])
lengths_batch shape: torch.Size([64])
Example X: tensor([1887,    8, 1161, 1162,  130,   36, 1163,    4,   26,  952,  294, 1151,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0])
Example y: tensor(1)
Example lengths: tensor(12)


In [42]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, X, lengths):
        embedded = self.embedding(X)

        outputs, last_hidden = self.rnn(embedded)

        batch_size = outputs.size(0)

        final_hidden = outputs[
            torch.arange(batch_size, device=outputs.device),
            lengths - 1
        ]

        logits = self.fc(final_hidden)

        return logits

In [44]:
vocab_size = len(vocab)
embedding_dim = 64
hidden_size = 128
num_classes = 2

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_classes=num_classes
).to(device)

X_batch, y_batch, lengths_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
lengths_batch = lengths_batch.to(device)

embedded = model.embedding(X_batch)
outputs, last_hidden = model.rnn(embedded)

batch_size = outputs.size(0)

# Correctly extract final_hidden using lengths_batch, as defined in SimpleRNNClassifier's forward method
final_hidden = outputs[
    torch.arange(batch_size, device=outputs.device),
    lengths_batch - 1
]

logits = model.fc(final_hidden)

print("X_batch shape:", X_batch.shape)
print("embedded shape:", embedded.shape)
print("outputs shape:", outputs.shape)
print("last_hidden shape:", last_hidden.shape)
print("final_hidden shape:", final_hidden.shape)
print("logits shape:", logits.shape)

X_batch shape: torch.Size([64, 30])
embedded shape: torch.Size([64, 30, 64])
outputs shape: torch.Size([64, 30, 128])
last_hidden shape: torch.Size([1, 64, 128])
final_hidden shape: torch.Size([64, 128])
logits shape: torch.Size([64, 2])


In [45]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_size=128,
    num_classes=2
).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [46]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch, lengths in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        lengths = lengths.to(device)

        logits = model(X_batch, lengths)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / total, correct / total

In [47]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths = lengths.to(device)

            logits = model(X_batch, lengths)
            loss = loss_fn(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / total, correct / total

In [48]:
n_epochs = 10

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"valid loss: {valid_loss:.4f} | "
        f"valid acc: {valid_acc:.4f}"
    )

Epoch 01 | train loss: 0.6929 | train acc: 0.5460 | valid loss: 0.6867 | valid acc: 0.5631
Epoch 02 | train loss: 0.6515 | train acc: 0.6186 | valid loss: 0.6970 | valid acc: 0.5573
Epoch 03 | train loss: 0.6092 | train acc: 0.6702 | valid loss: 0.6652 | valid acc: 0.6330
Epoch 04 | train loss: 0.5604 | train acc: 0.7122 | valid loss: 0.6502 | valid acc: 0.6227
Epoch 05 | train loss: 0.5026 | train acc: 0.7536 | valid loss: 0.6657 | valid acc: 0.6514
Epoch 06 | train loss: 0.4464 | train acc: 0.7922 | valid loss: 0.6853 | valid acc: 0.6617
Epoch 07 | train loss: 0.3843 | train acc: 0.8258 | valid loss: 0.7637 | valid acc: 0.6766
Epoch 08 | train loss: 0.3294 | train acc: 0.8568 | valid loss: 0.7618 | valid acc: 0.6617
Epoch 09 | train loss: 0.2756 | train acc: 0.8800 | valid loss: 0.8849 | valid acc: 0.6755
Epoch 10 | train loss: 0.2366 | train acc: 0.8994 | valid loss: 0.9191 | valid acc: 0.6342


In [49]:
def predict_sentiment(text, model, vocab, max_length, device):
    model.eval()

    ids = encode_text(text, vocab)
    length = min(len(ids), max_length)
    ids = pad_or_truncate(ids, max_length)

    X = torch.tensor([ids], dtype=torch.long).to(device)
    lengths = torch.tensor([length], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(X, lengths)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "positive" if pred == 1 else "negative"

    return label, probs.cpu()

In [50]:
sentences = [
    "this movie is amazing",
    "this movie is terrible",
    "i really enjoyed this film",
    "i did not like this movie",
    "a wonderful and emotional story",
    "boring and predictable"
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)

this movie is amazing → positive tensor([[0.0189, 0.9811]])
this movie is terrible → negative tensor([[0.9535, 0.0465]])
i really enjoyed this film → positive tensor([[0.1168, 0.8832]])
i did not like this movie → negative tensor([[0.7771, 0.2229]])
a wonderful and emotional story → positive tensor([[0.0089, 0.9911]])
boring and predictable → negative tensor([[0.9582, 0.0418]])


#Deep RNN

In [ ]:
!pip install datasets

In [2]:
from datasets import load_dataset

dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [3]:
print(dataset["train"][0])
print(dataset["train"][1])
print(dataset["train"][2])

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
{'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}
{'text': 'effective but too-tepid biopic', 'label': 1}


In [4]:
print(dataset["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [5]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

dataset = dataset.shuffle(seed=42)

train_data = dataset["train"].shuffle(seed=42)
valid_data = dataset["validation"].shuffle(seed=42)
test_data = dataset["test"].shuffle(seed=42)

print("Train labels:", Counter(train_data["label"]))
print("Valid labels:", Counter(valid_data["label"]))
print("Test labels:", Counter(test_data["label"]))

Train labels: Counter({1: 4265, 0: 4265})
Valid labels: Counter({0: 533, 1: 533})
Test labels: Counter({0: 533, 1: 533})


In [6]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b|[!?.,]", text.lower())


print(tokenize(train_data[0]["text"]))

['the', 'level', 'of', 'acting', 'elevates', 'the', 'material', 'above', 'pat', 'inspirational', 'status', 'and', 'gives', 'it', 'a', 'sturdiness', 'and', 'solidity', 'that', 'we', 've', 'long', 'associated', 'with', 'washington', 'the', 'actor', '.']


In [7]:
from collections import Counter

counter = Counter()

for item in train_data:
    tokens = tokenize(item["text"])
    counter.update(tokens)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

min_freq = 2

for word, count in counter.items():
    if count >= min_freq:
        vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 8803


In [8]:
max_length = 50

def encode_text(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return ids


def pad_or_truncate(ids, max_length):
    if len(ids) > max_length:
        ids = ids[:max_length]

    padded = ids + [vocab["<PAD>"]] * (max_length - len(ids))
    return padded

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader


class RottenTomatoesDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_length):
        self.dataset = hf_dataset
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["text"]
        label = self.dataset[idx]["label"]

        ids = encode_text(text, self.vocab)

        length = min(len(ids), self.max_length)
        ids = pad_or_truncate(ids, self.max_length)

        X = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        length = torch.tensor(length, dtype=torch.long)

        return X, y, length

In [10]:
train_set = RottenTomatoesDataset(train_data, vocab, max_length)
valid_set = RottenTomatoesDataset(valid_data, vocab, max_length)
test_set = RottenTomatoesDataset(test_data, vocab, max_length)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

In [11]:
X_batch, y_batch, lengths_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("lengths_batch shape:", lengths_batch.shape)
print("X_batch[0]:", X_batch[0])
print("y_batch[0]:", y_batch[0])
print("lengths_batch[0]:", lengths_batch[0])

X_batch shape: torch.Size([64, 50])
y_batch shape: torch.Size([64])
lengths_batch shape: torch.Size([64])
X_batch[0]: tensor([4019, 2225,   12,  401,  396,  416,  808, 4236,   46,  636,  862,  747,
          52, 2195, 1044,  173, 2783,  275,  202,   24,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0])
y_batch[0]: tensor(0)
lengths_batch[0]: tensor(20)


In [12]:
import torch
import torch.nn as nn


class DeepBiRNNTextClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size,
        num_classes,
        num_layers,
        dropout
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, X, lengths):
        embedded = self.embedding(X)

        outputs, last_hidden = self.rnn(embedded)

        batch_size = outputs.size(0)
        hidden_twice = outputs.size(2)
        hidden_size = hidden_twice // 2

        forward_last = outputs[
            torch.arange(batch_size, device=outputs.device),
            lengths - 1,
            :hidden_size
        ]

        backward_last = outputs[
            torch.arange(batch_size, device=outputs.device),
            0,
            hidden_size:
        ]

        final_hidden = torch.cat(
            [forward_last, backward_last],
            dim=1
        )

        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)

        return logits

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)

model = DeepBiRNNTextClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_size=128,
    num_classes=2,
    num_layers=2,
    dropout=0.3
).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [14]:
X_batch, y_batch, lengths_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
lengths_batch = lengths_batch.to(device)

embedded = model.embedding(X_batch)
outputs, last_hidden = model.rnn(embedded)

batch_size = outputs.size(0)
hidden_twice = outputs.size(2)
hidden_size = hidden_twice // 2

forward_last = outputs[
    torch.arange(batch_size, device=outputs.device),
    lengths_batch - 1,
    :hidden_size
]

backward_last = outputs[
    torch.arange(batch_size, device=outputs.device),
    0,
    hidden_size:
]

final_hidden = torch.cat(
    [forward_last, backward_last],
    dim=1
)

logits = model.fc(final_hidden)

print("X_batch shape:", X_batch.shape)
print("embedded shape:", embedded.shape)
print("outputs shape:", outputs.shape)
print("last_hidden shape:", last_hidden.shape)
print("forward_last shape:", forward_last.shape)
print("backward_last shape:", backward_last.shape)
print("final_hidden shape:", final_hidden.shape)
print("logits shape:", logits.shape)

X_batch shape: torch.Size([64, 50])
embedded shape: torch.Size([64, 50, 128])
outputs shape: torch.Size([64, 50, 256])
last_hidden shape: torch.Size([4, 64, 128])
forward_last shape: torch.Size([64, 128])
backward_last shape: torch.Size([64, 128])
final_hidden shape: torch.Size([64, 256])
logits shape: torch.Size([64, 2])


In [16]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch, lengths_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        lengths_batch = lengths_batch.to(device)

        logits = model(X_batch, lengths_batch)

        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [17]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch, lengths_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths_batch = lengths_batch.to(device)

            logits = model(X_batch, lengths_batch)

            loss = loss_fn(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [18]:
import copy

n_epochs = 15

best_valid_acc = 0
best_model_state = None

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        valid_loader,
        loss_fn,
        device
    )

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"valid loss: {valid_loss:.4f} | "
        f"valid acc: {valid_acc:.4f}"
    )

print("Best valid accuracy:", best_valid_acc)

model.load_state_dict(best_model_state)

Epoch 01 | train loss: 0.6957 | train acc: 0.5574 | valid loss: 0.6695 | valid acc: 0.6023
Epoch 02 | train loss: 0.6271 | train acc: 0.6484 | valid loss: 0.6569 | valid acc: 0.6295
Epoch 03 | train loss: 0.5439 | train acc: 0.7292 | valid loss: 0.6462 | valid acc: 0.6529
Epoch 04 | train loss: 0.4488 | train acc: 0.7946 | valid loss: 0.6585 | valid acc: 0.6632
Epoch 05 | train loss: 0.3507 | train acc: 0.8491 | valid loss: 0.7320 | valid acc: 0.6820
Epoch 06 | train loss: 0.2737 | train acc: 0.8852 | valid loss: 0.7454 | valid acc: 0.6782
Epoch 07 | train loss: 0.1916 | train acc: 0.9252 | valid loss: 0.9605 | valid acc: 0.6857
Epoch 08 | train loss: 0.1410 | train acc: 0.9462 | valid loss: 1.1840 | valid acc: 0.6914
Epoch 09 | train loss: 0.0965 | train acc: 0.9644 | valid loss: 1.3467 | valid acc: 0.6970
Epoch 10 | train loss: 0.0699 | train acc: 0.9753 | valid loss: 1.4822 | valid acc: 0.6998
Epoch 11 | train loss: 0.0560 | train acc: 0.9794 | valid loss: 1.5697 | valid acc: 0.7073

<All keys matched successfully>

In [19]:
test_loss, test_acc = evaluate(
    model,
    test_loader,
    loss_fn,
    device
)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

Test loss: 1.9425874722607812
Test accuracy: 0.7120075046904315


In [20]:
def predict_sentiment(text, model, vocab, max_length, device):
    model.eval()

    ids = encode_text(text, vocab)

    length = min(len(ids), max_length)

    ids = pad_or_truncate(ids, max_length)

    X = torch.tensor([ids], dtype=torch.long).to(device)
    lengths = torch.tensor([length], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(X, lengths)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "positive" if pred == 1 else "negative"

    return label, probs.cpu()

In [21]:
sentences = [
    "this movie was amazing",
    "this movie was terrible",
    "i really enjoyed this film",
    "i did not like this movie",
    "the film was boring and predictable",
    "the story was wonderful and emotional"
]

for sentence in sentences:
    label, probs = predict_sentiment(
        sentence,
        model,
        vocab,
        max_length,
        device
    )

    print(sentence, "→", label, probs)

this movie was amazing → negative tensor([[1.0000e+00, 6.0548e-07]])
this movie was terrible → negative tensor([[1.0000e+00, 7.5382e-09]])
i really enjoyed this film → positive tensor([[0.0040, 0.9960]])
i did not like this movie → negative tensor([[1.0000e+00, 7.9454e-08]])
the film was boring and predictable → negative tensor([[1.0000e+00, 3.1162e-08]])
the story was wonderful and emotional → positive tensor([[0.0014, 0.9986]])
